**Ch121a | Module 3: Periodic DFT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module3_Periodic-DFT/notebooks/05_aimd_and_mlff.ipynb)

# Notebook 5: Ab-initio MD & Machine-learned Force-fields

---

## Learning Objectives

- Understand what Ab-initio Molecular Dynamics (AIMD) is and how it differs from classical MD
- Compare Born-Oppenheimer MD and Car-Parrinello MD
- Know the key VASP tags for running AIMD simulations
- Understand the motivation for Machine-learned Force-fields (MLFFs) and how they bridge DFT accuracy with classical MD speed
- Recognize major MLFF architectures (Behler-Parrinello, GAP, DeePMD, NequIP, MACE)
- Know how to set up VASP's built-in ML-FF training workflow
- Appreciate universal potentials (MACE-MP-0, CHGNet, SevenNet) and their use cases
- Qualitatively compare Classical FF, AIMD, and MLFF in terms of accuracy, cost, and transferability

## 1. Ab-initio Molecular Dynamics (AIMD) with VASP

### What is AIMD?

In classical MD, atoms interact via empirical **force-fields** — parametrized functions fitted to experiment or quantum data. In **AIMD**, forces are computed *on-the-fly* from a DFT calculation at every time step. No force-field approximation is made.

$$F_I = -\nabla_{\mathbf{R}_I} E_{\text{DFT}}[\{\mathbf{R}\}]$$

The electronic structure problem is solved at each MD step, giving forces that are essentially exact within the chosen DFT approximation.

### Born-Oppenheimer MD vs. Car-Parrinello MD

| | Born-Oppenheimer MD (BOMD) | Car-Parrinello MD (CPMD) |
|---|---|---|
| Electronic ground state | Fully converged at each step | Propagated as fictitious classical degrees of freedom |
| SCF per step | Yes | No (electron mass parameter $\mu$) |
| Stability | Robust | Requires careful tuning of $\mu$ and time step |
| VASP support | ✅ Default | ❌ (use CPMD code) |

VASP implements **BOMD** exclusively (`IBRION=0`).

### Key VASP AIMD Tags

```
IBRION = 0       ! MD run (no ionic relaxation)
NSW    = 5000    ! Number of MD steps
POTIM  = 2.0     ! Time step in fs
TEBEG  = 300     ! Starting temperature (K)
TEEND  = 300     ! Ending temperature (K)  (same → NVT)
SMASS  = 0       ! Nosé–Hoover thermostat (0=Nosé mass auto)
MDALGO = 2       ! Nosé–Hoover chain thermostat
ISYM   = 0       ! Symmetry OFF for MD
```

The ionic equations of motion are integrated with the Verlet algorithm. The Nosé–Hoover thermostat (`MDALGO=2`) enforces NVT (canonical) ensemble sampling.

### Cost vs. Classical MD

Each AIMD step requires a DFT self-consistent field (SCF) cycle:

- **System size**: practical limit ~100–500 atoms (vs. millions for classical MD)
- **Timescale**: tens to hundreds of picoseconds (vs. microseconds for classical MD)
- **Cost**: roughly $\mathcal{O}(N^3)$ with system size $N$ for standard DFT

### When to Use AIMD

- Reaction mechanisms and free-energy barriers in solution
- Phase transitions and melting
- Liquid-state simulations (water, molten salts, liquid metals)
- Amorphous structure generation (melt-quench protocol)
- Vibrational spectroscopy (IR/Raman via dipole/polarizability autocorrelation)

### Example: Water at 300 K — INCAR snippet

```
# VASP AIMD INCAR for liquid water (NVT, 300 K)
SYSTEM  = Water 300K NVT
ISTART  = 0
ICHARG  = 2
ENCUT   = 400
EDIFF   = 1E-5
PREC    = Normal

# MD settings
IBRION  = 0
NSW     = 10000    ! 10000 steps × 0.5 fs = 5 ps
POTIM   = 0.5
TEBEG   = 300
TEEND   = 300
SMASS   = 0
MDALGO  = 2
ISYM    = 0

# DFT settings
GGA     = PE       ! PBE functional
IVDW    = 11       ! DFT-D3 dispersion (important for water!)
```

## 2. Machine-learned Force-fields (MLFFs) / MLIPs

### Motivation

AIMD is accurate but expensive. Classical force-fields are fast but not transferable. **MLFFs** aim to:

$$\underbrace{E_{\text{MLFF}}(\{\mathbf{R}\})}_{{\approx E_{\text{DFT}}}} \quad \text{at} \quad \underbrace{\text{classical MD cost}}_{{\ll \text{DFT cost}}}$$

The key insight: **atomic environments** $\mathcal{A}_i$ (local neighborhoods of atom $i$) encode most of the physics. The total energy is decomposed as:

$$E = \sum_i \varepsilon_i(\mathcal{A}_i)$$

A ML model learns $\varepsilon_i$ from a database of DFT reference calculations.

### Major MLFF Architectures

| Architecture | Year | Method | Key Feature |
|---|---|---|---|
| **Behler-Parrinello NN** | 2007 | Symmetry functions + feed-forward NN | First general NN potential |
| **GAP / SOAP** | 2010 | Gaussian process + SOAP descriptors | Uncertainty quantification |
| **DeePMD** | 2018 | Deep NN, frame-invariant descriptor | Scalable, GPU-accelerated |
| **NequIP** | 2022 | Equivariant GNN (E(3)) | Data-efficient, accurate |
| **Allegro** | 2023 | Local equivariant GNN | Massively parallel |
| **MACE** | 2023 | Higher-order ACE + equivariant GNN | State-of-the-art accuracy |
| **VASP ML-FF** | 2020+ | On-the-fly Bayesian (Gaussian) | Integrated in VASP 6 |

### Training Workflow

```
1. Collect DFT reference data
   (energies, forces, stresses for diverse structures)
         ↓
2. Train MLFF on reference database
   (minimize loss: ΔE, ΔF, Δσ)
         ↓
3. Validate on held-out DFT data
   (force RMSE < ~50 meV/Å is a common target)
         ↓
4. Run long MD with MLFF
   (ns–μs timescales, 10^3–10^6 atoms)
         ↓
5. Active learning: flag uncertain structures,
   recompute DFT, retrain → iterate
```

### VASP Built-in ML-FF

VASP 6 includes an on-the-fly MLFF based on moment tensor potentials trained via Bayesian regression. Key tags:

```
ML_LMLFF  = .TRUE.   ! Activate ML-FF module
ML_ISTART = 0        ! 0=train from scratch, 1=read existing model
                     ! 2=use existing model without retraining
IBRION    = 0        ! MD
NSW       = 50000    ! Long run; model improves early on, then runs fast
```

VASP alternates between DFT (when model is uncertain) and MLFF (when confident), building the training set automatically.

### Universal Potentials

Recent **foundation models** trained on large, diverse DFT datasets enable zero-shot or fine-tuned simulations across the periodic table:

- **MACE-MP-0** (2023): trained on Materials Project; covers 89 elements
- **CHGNet** (2023): includes charge/magnet degrees of freedom
- **SevenNet** (2024): efficient equivariant GNN, competitive accuracy

These can be used directly via Python packages (`mace-torch`, `chgnet`, `sevenn`) with ASE integration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Concept illustration: fitting a 1D PES with a simple NN surrogate ───
# This is a schematic — NOT an actual MLFF library call.
# We use numpy to mimic what a real MLFF training loop does.

np.random.seed(42)
rng = np.random.default_rng(42)

# True PES: Morse potential  V(r) = D_e*(1 - exp(-a*(r-r0)))^2
D_e, a, r0 = 5.0, 1.5, 1.5   # eV, Å^-1, Å
r = np.linspace(1.0, 4.0, 400)
V_true = D_e * (1 - np.exp(-a * (r - r0)))**2

# DFT 'reference' points (sparse, noisy)
r_dft = np.array([1.05, 1.2, 1.4, 1.6, 1.8, 2.1, 2.5, 3.0, 3.5, 3.9])
V_dft = D_e * (1 - np.exp(-a * (r_dft - r0)))**2 + rng.normal(0, 0.05, len(r_dft))

# Classical force-field: Lennard-Jones (deliberately wrong well depth)
eps_lj, sigma_lj = 3.0, 1.3
V_lj = 4*eps_lj * ((sigma_lj/r)**12 - (sigma_lj/r)**6)
V_lj = V_lj - V_lj.min()   # shift to zero

# 'MLFF' surrogate: polynomial fit to DFT points (mimics NN regression)
coeffs = np.polyfit(r_dft, V_dft, deg=8)
V_mlff = np.polyval(coeffs, r)

# ─── Plot ───
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(r, V_true,  'k-',  lw=2.5, label='True DFT PES (Morse)', zorder=3)
ax.scatter(r_dft, V_dft, color='royalblue', s=80, zorder=5,
           label='DFT reference points', edgecolors='navy')
ax.plot(r, V_mlff,  'tab:orange', lw=2, ls='--', label='MLFF (fitted to DFT)', zorder=4)
ax.plot(r, V_lj,    'tab:red',    lw=1.5, ls=':', label='Classical FF (Lennard-Jones)', zorder=2)

ax.set_xlim(1.0, 4.0)
ax.set_ylim(-0.5, 8)
ax.set_xlabel('Interatomic distance $r$ (Å)', fontsize=13)
ax.set_ylabel('Potential energy (eV)', fontsize=13)
ax.set_title('Schematic: DFT reference → MLFF fit vs. classical FF', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(0, color='gray', lw=0.5)
plt.tight_layout()
plt.show()

print('Key takeaway: MLFF (orange dashed) closely tracks the DFT PES (black).')
print('The classical FF (red dotted) captures the rough shape but misses accuracy.')
print('With enough DFT reference points, MLFF error → DFT error (< ~50 meV/Å).')

## 3. Summary: Comparing Simulation Methods

| Property | Classical FF | AIMD | MLFF / MLIP |
|---|:---:|:---:|:---:|
| **Accuracy** | Low–Medium | High (DFT-level) | High (≈ DFT) |
| **Computational cost** | Very low | Very high | Low–Medium |
| **System size** | 10⁶–10⁹ atoms | ~10²–10³ atoms | 10³–10⁶ atoms |
| **Timescale** | μs–ms | ~10–100 ps | ns–μs |
| **Transferability** | Poor (empirical) | Excellent | Good (within training domain) |
| **Training required** | No (parameterized) | No | Yes (DFT database) |
| **Uncertainty quantification** | None | Implicit (DFT error) | Possible (GAP, VASP ML-FF) |
| **Open-source codes** | LAMMPS, GROMACS | VASP, CP2K, Quantum ESPRESSO | MACE, DeePMD-kit, NequIP, SchNetPack |

---

## 4. Further Reading

- Behler & Parrinello, *PRL* **98**, 146401 (2007) — first NN potential
- Bartók et al., *PRL* **104**, 136403 (2010) — GAP/SOAP
- Zhang et al., *PRL* **120**, 143001 (2018) — DeePMD
- Batzner et al., *Nat. Commun.* **13**, 2453 (2022) — NequIP
- Batatia et al., *NeurIPS* (2022) — MACE
- Batatia et al., *arXiv* 2401.00096 (2024) — MACE-MP-0 universal potential
- Kresse & Hafner, *Phys. Rev. B* **47**, 558 (1993) — VASP AIMD original paper
- [VASP Wiki: ML_LMLFF](https://www.vasp.at/wiki/index.php/ML_LMLFF)

---
*Ch121a | Caltech | Module 3 — Notebook 5 of 5*